## MODELS OPTIMIZATION AND MLFLOW

In [ ]:
import os
import joblib
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Setting up mlflow tracking uri & experiment title
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Customer_Churn_Optimization")

# Load data
train_df = pd.read_csv("../data/processed/train_resampled_smote.csv")
test_df = pd.read_csv("../data/processed/test_data.csv")

X_train_resampled = train_df.drop(columns=["churn"])
y_train_resampled = train_df["churn"]

X_test = test_df.drop(columns=["churn"])
y_test = test_df["churn"]

# Baseline Model Evaluation
print("Evaluating Baseline Model...")
base_model = XGBClassifier(eval_metric="logloss", random_state=42)
base_model.fit(X_train_resampled, y_train_resampled)

y_pred_base = base_model.predict(X_test)
y_prob_base = base_model.predict_proba(X_test)[:, 1]

base_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_base),
    "precision": precision_score(y_test, y_pred_base, zero_division=0),
    "recall": recall_score(y_test, y_pred_base, zero_division=0),
    "f1_score": f1_score(y_test, y_pred_base, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_prob_base)
}

print("Baseline Metrics:")
for k, v in base_metrics.items():
    print(f"  - {k}: {v:.4f}")

# Hyperparameter Tuning & MLflow Logging
print("\nStarting Hyperparameter Tuning...")
param_distributions = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [3, 5, 7, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.8, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=XGBClassifier(eval_metric="logloss", random_state=42),
    param_distributions=param_distributions,
    n_iter=5,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1
)

with mlflow.start_run(run_name="XGBoost_Tuned_Optimization"):
    mlflow.set_tag("model_type", "XGBoost")
    mlflow.set_tag("phase", "hyperparameter_tuning")
    
    random_search.fit(X_train_resampled, y_train_resampled)
    
    best_model = random_search.best_estimator_
    best_params = random_search.best_params_
    
    y_pred_tuned = best_model.predict(X_test)
    y_prob_tuned = best_model.predict_proba(X_test)[:, 1]
    
    tuned_metrics = {
        "accuracy": accuracy_score(y_test, y_pred_tuned),
        "precision": precision_score(y_test, y_pred_tuned, zero_division=0),
        "recall": recall_score(y_test, y_pred_tuned, zero_division=0),
        "f1_score": f1_score(y_test, y_pred_tuned, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob_tuned)
    }
    
    mlflow.log_params(best_params)
    mlflow.log_metrics({f"tuned_{k}": v for k, v in tuned_metrics.items()})
    mlflow.log_metrics({f"baseline_{k}": v for k, v in base_metrics.items()})
    
    mlflow.sklearn.log_model(
        best_model, 
        "tuned_churn_model", 
        skops_trusted_types=["xgboost.sklearn.XGBClassifier", "xgboost.core.Booster"]
    )
    
    print("\nBest Parameters Found:")
    print(best_params)
    
    print("\nTuned Metrics:")
    print(f"  - F1-Score: {tuned_metrics['f1_score']:.4f}")
    print(f"  - ROC-AUC: {tuned_metrics['roc_auc']:.4f}")

#  Save Model Locally
os.makedirs("../models/classification", exist_ok=True)
model_path = "../models/classification/churn_model.pkl"
joblib.dump(best_model, model_path)
print(f"\nSuccessfully saved the best model locally to: {model_path}")

MlflowException: The filesystem tracking backend (e.g., './mlruns') is in maintenance mode and will not receive further updates. Please migrate to a database backend (e.g., 'sqlite:///mlflow.db') to access the latest MLflow features. The `mlflow migrate-filestore` tool migrates your existing data losslessly. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance. If the filesystem backend is required for your workflow, set `MLFLOW_ALLOW_FILE_STORE=true` to opt out of this exception.